# Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import re
import random
from torch.utils.data import Dataset, DataLoader

# Generate Dataset

In [ ]:
greetings = [
    ("hello", "hi"),
    ("hi", "hello"),
    ("hey", "hey there"),
    ("good morning", "good morning"),
]

how_are_you = [
    ("how are you", "i am fine"),
    ("how are you doing", "i am doing well"),
]

identity = [
    ("what is your name", "i am a chatbot"),
    ("who are you", "i am your assistant"),
]

farewell = [
    ("bye", "goodbye"),
    ("see you", "see you later"),
]

questions = [
    ("what is ai", "ai is artificial intelligence"),
    ("where do you live", "i live on the internet"),
]

base_pairs = greetings + how_are_you + identity + farewell + questions

pairs = []
for _ in range(1000):
    q, a = random.choice(base_pairs)

    variations = [
        q,
        q + " ?",
        "can you tell me " + q,
        "please tell me " + q,
    ]

    pairs.append((random.choice(variations), a))

print("Dataset size:", len(pairs))

# Preprocessing

In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9]+", " ", text)
    return text.strip()

pairs = [(clean_text(q), clean_text(a)) for q, a in pairs]
pairs[:5]

[('please tell me what is ai', 'ai is artificial intelligence'),
 ('good morning', 'good morning'),
 ('what is ai', 'ai is artificial intelligence'),
 ('can you tell me hey', 'hey there'),
 ('good morning', 'good morning')]

# Vocabulary

In [ ]:
class Vocab:
    def __init__(self):
        self.word2idx = {"<pad>":0, "<sos>":1, "<eos>":2}
        self.idx2word = {0:"<pad>", 1:"<sos>", 2:"<eos>"}
        self.count = 3

    def add_sentence(self, sentence):
        for word in sentence.split():
            if word not in self.word2idx:
                self.word2idx[word] = self.count
                self.idx2word[self.count] = word
                self.count += 1

vocab = Vocab()

for q, a in pairs:
    vocab.add_sentence(q)
    vocab.add_sentence(a)

print("Vocab size:", vocab.count)

# Tensor Conversion

In [5]:
def sentence_to_tensor(sentence, vocab):
    tokens = [vocab.word2idx[word] for word in sentence.split()]
    tokens = [vocab.word2idx["<sos>"]] + tokens + [vocab.word2idx["<eos>"]]
    return torch.tensor(tokens)

data = [(sentence_to_tensor(q, vocab), sentence_to_tensor(a, vocab)) for q, a in pairs]

# Model Architecture

In [6]:
class Encoder(nn.Module):
    def __init__(self, input_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(input_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size)

    def forward(self, x):
        embedded = self.embedding(x).unsqueeze(1)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

# Attention

In [7]:
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.shape[0]
        hidden = hidden.repeat(seq_len, 1, 1)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=0)

# Decoder

In [11]:
class Decoder(nn.Module):
    def __init__(self, output_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(output_size, embed_size)
        self.lstm = nn.LSTM(hidden_size + embed_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)
        self.attention = Attention(hidden_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        x = x.unsqueeze(0)
        embedded = self.embedding(x).unsqueeze(1)
        attn_weights = self.attention(hidden, encoder_outputs)
        attn_weights = attn_weights.unsqueeze(1)


        context = torch.sum(attn_weights * encoder_outputs, dim=0)
        context = context.unsqueeze(0)


        lstm_input = torch.cat((embedded, context), dim=2)

        outputs, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        predictions = self.fc(outputs.squeeze(0))
        return predictions, hidden, cell

# Training Setup

In [ ]:
input_size = vocab.count
output_size = vocab.count

encoder = Encoder(input_size, 64, 128)
decoder = Decoder(output_size, 64, 128)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)

device = torch.device("cpu")
encoder.to(device)
decoder.to(device)

# Training Loop

In [ ]:
for epoch in range(50):
    total_loss = 0

    for src, trg in data[:300]:
        optimizer.zero_grad()
        src = src.to(device)
        trg = trg.to(device)

        encoder_outputs, hidden, cell = encoder(src)
        input_token = trg[0]

        loss = 0

        for t in range(1, len(trg)):
            output, hidden, cell = decoder(input_token, hidden, cell, encoder_outputs)
            loss += criterion(output, trg[t].unsqueeze(0))
            input_token = trg[t]

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

# Evaluation

In [19]:
def sentence_to_tensor(sentence, vocab):
    tokens = [vocab.word2idx[word] for word in sentence.split()]
    tokens = [vocab.word2idx["<sos>"]] + tokens + [vocab.word2idx["<eos>"]]
    return torch.tensor(tokens, dtype=torch.long)


def generate_response(sentence):
    encoder.eval()
    decoder.eval()

    sentence = clean_text(sentence)

    src = sentence_to_tensor(sentence, vocab).to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = encoder(src)

        input_token = torch.tensor([vocab.word2idx["<sos>"]], device=device)
        result = []

        for _ in range(10):
            output, hidden, cell = decoder(input_token, hidden, cell, encoder_outputs)
            predicted = output.argmax(1).item()

            if predicted == vocab.word2idx["<eos>"]:
                break

            result.append(vocab.idx2word[predicted])
            input_token = torch.tensor([predicted], device=device)

    return " ".join(result)